# Baseline Balance: Standardised Differences`
'02_customer_profile_group_comparison.ipynb' notes that *"descriptive similarity alone is notsufficient to establish comparability"* and that statistical assessment should follow before a final conclusion on baseline balance.This notebook adds that step. It rebuilds the same experimental population used there, reproduces thedescriptive group summary as a reference point, and then quantifies the gap between Test and Control with standardised mean differences.

## Setup — same population as in 02

In [8]:
import pandas as pd

demo_df = pd.read_csv("../data/raw/df_final_demo.txt")
experiment_df = pd.read_csv("../data/raw/df_final_experiment_clients (1).txt")

clients_df = demo_df.merge(experiment_df, on="client_id", how="inner")

ab_clients_df = clients_df[clients_df["Variation"].isin(["Test", "Control"])].copy()

print("merged clients:", clients_df.shape)
print("A/B experiment clients:", ab_clients_df.shape)
ab_clients_df["Variation"].value_counts()

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.
merged clients: (70609, 10)
A/B experiment clients: (50500, 10)


Variation
Test       26968
Control    23532
Name: count, dtype: int64

#### 50_500 clients with an assigned variation:
#### -> 26,968 Test and 23,532 Control
#### -> difference between the two groups = 3400 clients, which is what makes the next step necessary

## 1. Why descriptive comparison is not enough? 
The group summary below is the one already produced in '02'. It is reproduced here as the reference point, not as a new result.

In [9]:
baseline = ["clnt_age", "clnt_tenure_mnth", "num_accts", "bal", "calls_6_mnth", "logons_6_mnth",]

ab_clients_df.groupby("Variation")[baseline].agg(["mean", "median"]).round(2)

clnt_age        clnt_tenure_mnth        num_accts               bal  \
              mean median             mean median      mean median       mean   
Variation                                                                       
Control      47.50   48.5           151.06  137.0      2.26    2.0  150147.33   
Test         47.16   47.5           149.85  134.0      2.25    2.0  148962.61   

                    calls_6_mnth        logons_6_mnth         
             median         mean median          mean median  
Variation                                                     
Control    66024.18         3.13    3.0          6.17    6.0  
Test       65468.36         3.06    3.0          6.10    6.0

#### The averages look close
#### However "close" depends on how spread out each variable is —> a one-year gap in age isn't the same thing as a one-year gap in tenure

#### The standardised difference investigate that -> it divides the gap between the two groups by the standard deviation of the variable. #### Below 0.1 is the usual threshold for calling two groups balanced

-----

## 2. Standardised differences for the numeric baseline variables

In [10]:
test = ab_clients_df[ab_clients_df["Variation"] == "Test"]
control = ab_clients_df[ab_clients_df["Variation"] == "Control"]

balance = pd.DataFrame({
    "test_mean":    test[baseline].mean(),
    "control_mean": control[baseline].mean(),
})

pooled_sd = ((test[baseline].std() ** 2 + control[baseline].std() ** 2) / 2) ** 0.5

balance["smd"] = (balance["test_mean"] - balance["control_mean"]) / pooled_sd
balance["balanced"] = balance["smd"].abs() < 0.1

balance.round(3)

,test_mean,control_mean,smd,balanced
clnt_age,47.164,47.498,-0.022,True
clnt_tenure_mnth,149.853,151.060,-0.015,True
num_accts,2.250,2.260,-0.019,True
bal,148962.605,150147.327,-0.004,True
calls_6_mnth,3.062,3.129,-0.031,True
logons_6_mnth,6.102,6.166,-0.030,True


#### A value of 0.02 means the two groups differ by two hundredths of a standard deviation on that characteristic
#### = far below the 0.1 threshold, and far below anything that could influence the outcome of the experiment

-----

## 3. Same check for gender (categorical variable)
The difference is measured between the proportions in each group rather than between means. 

The logic is identical -> the gap is divided by the pooled spread of the proportion.

In [11]:
# split by experimental group
test = ab_clients_df[ab_clients_df["Variation"] == "Test"]
control = ab_clients_df[ab_clients_df["Variation"] == "Control"]

# gender is transformed into True/False columns, one per category
categories = sorted(ab_clients_df["gendr"].dropna().unique())

test_gender = pd.get_dummies(test["gendr"]).reindex(columns=categories, fill_value=False)
control_gender = pd.get_dummies(control["gendr"]).reindex(columns=categories, fill_value=False)

# the mean of a True/False column is the share of that category in the group
p_test = test_gender.mean()
p_control = control_gender.mean()

gender_balance = pd.DataFrame({
    "test_pct":    p_test * 100,
    "control_pct": p_control * 100,
})

# for proportions the spread is p * (1 - p), not the standard deviation
pooled_sd = ((p_test * (1 - p_test) + p_control * (1 - p_control)) / 2) ** 0.5

gender_balance["smd"] = (p_test - p_control) / pooled_sd
gender_balance["balanced"] = gender_balance["smd"].abs() < 0.1

gender_balance.round(3)

,test_pct,control_pct,smd,balanced
F,32.320,32.054,0.006,True
M,33.288,33.869,-0.012,True
U,34.359,34.056,0.006,True
X,0.007,0.000,0.012,True


------

## 4. Assumptions and statistical tests

The checklist asks for statistical tests and for their assumptions to be verified. Before choosing a test, we need to know whether the baseline variables are normally distributed

In [12]:
from scipy import stats

pd.DataFrame({
    "skew":     ab_clients_df[baseline].skew(),
    "kurtosis": ab_clients_df[baseline].kurtosis(),
}).round(2)

,skew,kurtosis
clnt_age,0.08,-0.95
clnt_tenure_mnth,1.05,1.05
num_accts,2.29,6.00
bal,11.47,294.12
calls_6_mnth,0.05,-1.44
logons_6_mnth,0.03,-1.43


#### A t-test assumes roughly normal distributions. Balance is heavily right-skewed and the count variables aren't normal either
#### so that assumption doesn't hold
#### Mann-Whitney compares the two groups without requiring it

In [16]:
p_values = {}
for col in baseline:
    stat, p = stats.mannwhitneyu(test[col].dropna(), control[col].dropna())
    p_values[col] = p

balance["p_value"] = pd.Series(p_values)
balance.round(3)

,test_mean,control_mean,smd,balanced,p_value
clnt_age,47.164,47.498,-0.022,True,0.015
clnt_tenure_mnth,149.853,151.060,-0.015,True,0.088
num_accts,2.250,2.260,-0.019,True,0.030
bal,148962.605,150147.327,-0.004,True,0.149
calls_6_mnth,3.062,3.129,-0.031,True,0.001
logons_6_mnth,6.102,6.166,-0.030,True,0.001


In [17]:
contingency = pd.crosstab(ab_clients_df["gendr"], ab_clients_df["Variation"])
chi2, p, dof, expected = stats.chi2_contingency(contingency)

print(f"chi-square p-value: {p:.3f}")
print("smallest expected count:", expected.min().round(1))

chi-square p-value: 0.305
smallest expected count: 0.9


-----

## 5. Why the SMD leads and the p-values follow

the tests above agree with the standardised differences, but the SMD is what the conclusion rests on:

-> with 50_500 clients, a test flags a two-month gap in average tenure as significant -> it is real, BUT IRRELEVANT
-> testing seven variables means one will look significant by chance
-> and not rejecting H0 doesn't prove the groups are the same, which is the thing we want to know

#### the p-values answer "could this gap be chance?"; the SMD answers "is the gap big enough to matter?" - and for baseline balance, that's the question

-----

### Conclusion: Are Test and Control comparable?

**Yes** 

Every standardised difference is below 0.1, and the gender split differs by fractions of a percentage point. Notebook 02 reached the same conclusion descriptively —> this puts a number on it.

#### Two things worth noting:
#### - we can only check what we measured. Digital confidence or motivation aren't in the data, a randomisation is what covers those.
#### - this is a baseline check, not an outcome check. Completion rate or session duration are supposed to differ if the redesign works.